In [1]:
"""
Step 2: Preprocessing and Trial Extraction for BCI Competition IV-2a
- Filters: 8-30 Hz bandpass (Sensorimotor rhythm: Mu + Beta)
- Classes: Left Hand (0) vs Right Hand (1)
- Epoch: t = 2.0s to 6.0s (active MI window: 4 seconds @ 250 Hz -> Exactly 1000 timepoints)
- Channels: 22 EEG channels (EOG excluded)
- Output: Standard .npy files + dataset_metadata_bci2a.csv
"""

import os
from pathlib import Path
import numpy as np
import pandas as pd
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery


def preprocess_bci2a(output_dir=os.path.join("..", "data_bci2a_processed_1000")):
    output_path = Path(output_dir)
    npy_dir = output_path / "npy_trials"
    npy_dir.mkdir(parents=True, exist_ok=True)

    print(">>> Initializing MOABB Paradigm for 2-class MI (8-30 Hz)...")

    # Paradigm configuration strictly aligned with the study:
    # 2 classes: left_hand vs right_hand
    # 8-30 Hz bandpass (active sensorimotor rhythms)
    # 4-second epoch after cue (2.0s to 6.0s)
    paradigm = MotorImagery(
        events=["left_hand", "right_hand"],
        n_classes=2,
        fmin=8.0,
        fmax=30.0,
        tmin=2.0,
        tmax=6.0,
        resample=250,  # Native sampling rate
    )

    dataset = BNCI2014_001()
    subjects = list(range(1, 10))

    print(
        f">>> Extracting trials across all 9 subjects using native MOABB pipeline..."
    )
    # X shape: (n_trials, n_channels, n_timepoints)
    # y shape: (n_trials,)
    # meta: DataFrame with subject, session, run info
    X, y, meta = paradigm.get_data(dataset=dataset, subjects=subjects)

    print(f"[*] Raw extraction completed! Initial shape: {X.shape}")

    # Enforce an exact 4-second window: exactly 1000 samples at 250 Hz
    if X.shape[2] > 1000:
        X = X[:, :, :1000]
        print(
            f"[*] Trimmed timepoints from {X.shape[2] + 1} to exact 1000 samples."
        )
    elif X.shape[2] < 1000:
        raise ValueError(
            f"Unexpected number of timepoints: {X.shape[2]}. At least 1000 samples are required."
        )

    print(f"    Final Tensor Shape: Channels={X.shape[1]} | Timepoints={X.shape[2]}")

    # Encode labels: left_hand -> 0, right_hand -> 1
    label_map = {"left_hand": 0, "right_hand": 1}
    y_int = np.array([label_map[label] for label in y], dtype=np.int32)

    records = []
    print(">>> Saving individual trial .npy files...")

    for i in range(len(X)):
        trial_data = X[i].astype(
            np.float32
        )  # Shape: (22, 1000) -> (Channels, Timepoints)
        sub_id = int(meta.iloc[i]["subject"])
        trial_filename = f"trial_sub{sub_id:02d}_{i:04d}.npy"
        trial_path = npy_dir / trial_filename

        np.save(trial_path, trial_data)

        records.append(
            {
                "file": str(trial_path.as_posix()),
                "label": int(y_int[i]),
                "subject_id": sub_id,
            }
        )

    # Save metadata CSV
    metadata_df = pd.DataFrame(records)
    csv_path = output_path / "dataset_metadata_bci2a.csv"
    metadata_df.to_csv(csv_path, index=False)

    print("\n" + "=" * 55)
    print(">>> PREPROCESSING COMPLETE!")
    print(f"Total Trials Saved: {len(metadata_df)}")
    print(f"Metadata CSV saved to: {csv_path}")
    print("Class distribution:\n", metadata_df["label"].value_counts())
    print("Trials per subject:\n", metadata_df["subject_id"].value_counts())
    print("=" * 55)


def validate_dataset(output_dir=os.path.join("..", "data_bci2a_processed_1000")):
    print("\n>>> Running Post-Preprocessing Validation...")
    output_path = Path(output_dir)
    metadata_path = output_path / "dataset_metadata_bci2a.csv"

    if not metadata_path.exists():
        raise FileNotFoundError(f"Metadata file not found at {metadata_path}")

    metadata = pd.read_csv(metadata_path)

    print(f"Total trials in CSV: {len(metadata)}")
    print("\nClass distribution:")
    print(metadata["label"].value_counts().sort_index())

    print("\nTrials per subject:")
    print(metadata["subject_id"].value_counts().sort_index())

    # Load first sample file to verify dimensions and type
    sample_record = metadata.iloc[0]
    sample_path = Path(sample_record["file"])
    sample_data = np.load(sample_path)

    print(f"\nVerifying sample file: {sample_path.name}")
    print(f"    Shape: {sample_data.shape} (Expected: (22, 1000))")
    print(f"    Dtype: {sample_data.dtype} (Expected: float32)")
    print(f"    Subject ID: {sample_record['subject_id']}")
    print(f"    Label: {sample_record['label']}")

    assert (
        sample_data.shape == (22, 1000)
    ), f"Error: Shape mismatch {sample_data.shape}"
    assert (
        sample_data.dtype == np.float32
    ), f"Error: Dtype mismatch {sample_data.dtype}"
    assert metadata["subject_id"].nunique() == 9, "Error: Must have 9 subjects."

    print("\n>>> VALIDATION SUCCESSFUL: All checks passed!")


if __name__ == "__main__":
    preprocess_bci2a()
    validate_dataset()

<frozen importlib._bootstrap>:219: FutureWarning: mne.io.pick.pick_types is deprecated will be removed in 1.6, use documented public API instead. If no appropriate public API exists, please open an issue on GitHub.
<frozen importlib._bootstrap>:219: FutureWarning: mne.io.pick.pick_channels_regexp is deprecated will be removed in 1.6, use documented public API instead. If no appropriate public API exists, please open an issue on GitHub.
<frozen importlib._bootstrap>:219: FutureWarning: mne.io.pick.channel_type is deprecated will be removed in 1.6, use documented public API instead. If no appropriate public API exists, please open an issue on GitHub.
<python-env>/lib\site-packages\moabb\pipelines\__init__.py:26: ModuleNotFoundError: Tensorflow is not installed. You won't be able to use these MOABB pipelines if you attempt to do so.
  warn(


To use the get_shape_from_baseconcar, InputShapeSetterEEG, BraindecodeDatasetLoaderyou need to install `braindecode`.`pip install braindecode` or Please refer to `https://braindecode.org`.
>>> Initializing MOABB Paradigm for 2-class MI (8-30 Hz)...
>>> Extracting trials across all 9 subjects using native MOABB pipeline...


<python-env>/lib\site-packages\sklearn\preprocessing\_function_transformer.py:312: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  return func(X, **(kw_args if kw_args else {}))
<python-env>/lib\site-packages\sklearn\preprocessing\_function_transformer.py:312: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  return func(X, **(kw_args if kw_args else {}))
<python-env>/lib\site-packages\sklearn\preprocessing\_function_transformer.py:312: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  return func(X, **(kw_args if kw_args else {}))
<python-env>/lib\site-packages\sklearn\preprocessing\_function_transformer.py:312: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explic

[*] Raw extraction completed! Initial shape: (2538, 22, 1001)
[*] Trimmed timepoints from 1001 to exact 1000 samples.
    Final Tensor Shape: Channels=22 | Timepoints=1000
>>> Saving individual trial .npy files...

>>> PREPROCESSING COMPLETE!
Total Trials Saved: 2538
Metadata CSV saved to: data_bci2a_processed_1000\dataset_metadata_bci2a.csv
Class distribution:
 1    1294
0    1244
Name: label, dtype: int64
Trials per subject:
 6    283
9    283
2    282
3    282
5    282
7    282
8    282
1    281
4    281
Name: subject_id, dtype: int64

>>> Running Post-Preprocessing Validation...
Total trials in CSV: 2538

Class distribution:
0    1244
1    1294
Name: label, dtype: int64

Trials per subject:
1    281
2    282
3    282
4    281
5    282
6    283
7    282
8    282
9    283
Name: subject_id, dtype: int64

Verifying sample file: trial_sub01_0000.npy
    Shape: (22, 1000) (Expected: (22, 1000))
    Dtype: float32 (Expected: float32)
    Subject ID: 1
    Label: 1

>>> VALIDATION SUCCESSF